# Intro to MicroFVS
This notebook provides a gentle, but relatively thorough, introduction to MicroFVS. MicroFVS is designed to provide access to forest growth-and-yield capabilities using the Forest Vegetation Simulator through a REST API.

We will step through three fundamental components of using MicroFVS in this notebook:
1. Forest inventory input data
2. Keyfiles to define FVS simulation instructions
3. Submitting and receiving requests from the MicroFVS web service

In [1]:
import json
import re

import pandas as pd
import requests
from pydantic import ValidationError

from microfvs.enums import (
    FvsEventType,
    FvsKeyfileTemplate,
    FvsOutputTableName,
    FvsVariant,
)
from microfvs.models import (
    FvsEventLibrary,
    FvsKeyfile,
    FvsResult,
    FvsStandInit,
    FvsTreeInit,
    FvsTreeInitRecord,
)

## 1. Forest Inventory Inputs

Here we will cover the core data formats that will allow us to create or load forest inventory data for use in simulations, and to ensure that it passes some basic quality assurance checks. Our primary building blocks for forest inventory data are: `FvsStandInit`, `FvsTreeInitRecord`, and `FvsTreeInit`.

MicroFVS currently provides for the execution of a single simulation for a single stand as the minimal starting point. This starting point is intended to be extended to support batch modeling capabilities soon.

`FvsStandInit` contains the attributes of a single stand.

In [2]:
FvsStandInit?

Init signature:
FvsStandInit(
    *,
    stand_id: str,
    variant: microfvs.enums.FvsVariant,
    inv_year: int,
    basal_area_factor: float,
    inv_plot_size: float,
    brk_dbh: float,
    stand_cn: str | None = None,
    latitude: float | None = None,
    longitude: float | None = None,
    region: int | None = None,
    forest: int | None = None,
    district: int | None = None,
    location: int | None = None,
    ecoregion: str | None = None,
    pv_code: int | None = None,
    habitat: int | str | None = None,
    pv_ref_code: int | None = None,
    age: int | None = None,
    aspect: float | None = None,
    slope: float | None = None,
    elevft: float | None = None,
    elevation: float | None = None,
    num_plots: int | None = None,
    nonstk_plots: int | None = None,
    sam_wt: float | None = None,
    stk_pcnt: float | None = None,
    dg_trans: int | None = None,
    dg_measure: int | None = None,
    htg_trans: int | None = None,
    htg_measure: int | None = None

Let's make a new `StandInitRecord` with the minimum required fields ...

In [3]:
stand_init = FvsStandInit(
    stand_id="12345",
    variant="PN",
    inv_year=2020,
    basal_area_factor=-10,  # 1/10th-acre fixed radius plot for large trees
    inv_plot_size=100,  # 1/100th-acre fixed radius plot for small trees
    brk_dbh=5,  # large trees start at 5" DBH
)
stand_init

FvsStandInit(stand_id='12345', variant='PN', inv_year=2020, basal_area_factor=-10.0, inv_plot_size=100.0, brk_dbh=5.0, stand_cn=None, latitude=None, longitude=None, region=None, forest=None, district=None, location=None, ecoregion=None, pv_code=None, habitat=None, pv_ref_code=None, age=None, aspect=None, slope=None, elevft=None, elevation=None, num_plots=None, nonstk_plots=None, sam_wt=None, stk_pcnt=None, dg_trans=None, dg_measure=None, htg_trans=None, htg_measure=None, mort_measure=None, max_ba=None, max_sdi=None, site_species=None, site_index=None, model_type=None, physio_region=None, forest_type=None, state=None, county=None, fuel_model=None, fuel_0_25_h=None, fuel_25_1_h=None, fuel_1_3_h=None, fuel_3_6_h=None, fuel_6_12_h=None, fuel_12_20_h=None, fuel_20_35_h=None, fuel_35_50_h=None, fuel_gt_50_h=None, fuel_0_25_s=None, fuel_25_1_s=None, fuel_1_3_s=None, fuel_3_6_s=None, fuel_6_12_s=None, fuel_12_20_s=None, fuel_20_35_s=None, fuel_35_50_s=None, fuel_gt_50_s=None, fuel_litter=None,

At the tree-level, our equivalent minimal data container is an `FvsTreeInitRecord`. This stores the attributes of a single tree.

In [4]:
FvsTreeInitRecord?

Init signature:
FvsTreeInitRecord(
    *,
    stand_id: str,
    plot_id: int | None = None,
    tree_id: int | str | None = None,
    tree_count: float | None = None,
    species: int | str,
    diameter: typing.Annotated[float, Gt(gt=0)],
    history: int | None = None,
    ht: typing.Annotated[float | None, Gt(gt=0)] = None,
    crratio: typing.Annotated[int | None, Ge(ge=0), Le(le=100)] = None,
    dg: float | None = None,
    htg: float | None = None,
    httopk: typing.Annotated[float | None, Ge(ge=0)] = None,
    damage1: typing.Annotated[int | None, Ge(ge=1), Le(le=99)] = None,
    severity1: typing.Annotated[int | None, Ge(ge=0), Le(le=100)] = None,
    damage2: typing.Annotated[int | None, Ge(ge=1), Le(le=99)] = None,
    severity2: typing.Annotated[int | None, Ge(ge=0), Le(le=100)] = None,
    damage3: typing.Annotated[int | None, Ge(ge=1), Le(le=99)] = None,
    severity3: typing.Annotated[int | None, Ge(ge=0), Le(le=100)] = None,
    tree_value: typing.Annotated[int | None

We combine multiple `FvsTreeInitRecord`s into an `FvsTreeInit` object, which represents the trees present within a single stand we intend to use for simulation.

In [5]:
tree1 = FvsTreeInitRecord(
    stand_id="12345", plot_id=1, tree_id=1, species="DF", diameter=10.5
)
tree2 = FvsTreeInitRecord(
    stand_id="12345", plot_id=1, tree_id=2, species=242, dbh=12.0
)
seed1 = FvsTreeInitRecord(
    stand_id="12345", plot_id=1, tree_id=3, species=263, dbh=0.1, tree_count=2
)
tree_init = FvsTreeInit(trees=[tree1, tree2, seed1])
tree_init

FvsTreeInit(trees=[FvsTreeInitRecord(stand_id='12345', plot_id=1, tree_id=1, tree_count=None, species='DF', diameter=10.5, history=None, ht=None, crratio=None, dg=None, htg=None, httopk=None, damage1=None, severity1=None, damage2=None, severity2=None, damage3=None, severity3=None, tree_value=None, prescription=None, slope=None, aspect=None, habitat=None, topo_code=None, site_prep=None, age=None), FvsTreeInitRecord(stand_id='12345', plot_id=1, tree_id=2, tree_count=None, species=242, diameter=12.0, history=None, ht=None, crratio=None, dg=None, htg=None, httopk=None, damage1=None, severity1=None, damage2=None, severity2=None, damage3=None, severity3=None, tree_value=None, prescription=None, slope=None, aspect=None, habitat=None, topo_code=None, site_prep=None, age=None), FvsTreeInitRecord(stand_id='12345', plot_id=1, tree_id=3, tree_count=2.0, species=263, diameter=0.1, history=None, ht=None, crratio=None, dg=None, htg=None, httopk=None, damage1=None, severity1=None, damage2=None, severi

These `FvsTreeInitRecord`, `FvsTreeInit` and `FvsStandInit` classes are implemented as [Pydantic BaseModels](https://docs.pydantic.dev/latest/), so they come along with type-checking and other validation settings that will tell you when you've done something that really won't work. These checks are not exhaustive (e.g., checking for species codes FVS doesn't recognize, checking that your trees aren't wider than they are tall, or that the field crew used the wrong side of the D-tape).

In [6]:
try:  # missing several required fields
    FvsStandInit(stand_id="12345")
except ValidationError as e:
    print(e)

5 validation errors for FvsStandInit
variant
  Field required [type=missing, input_value={'stand_id': '12345'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
inv_year
  Field required [type=missing, input_value={'stand_id': '12345'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
basal_area_factor
  Field required [type=missing, input_value={'stand_id': '12345'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
inv_plot_size
  Field required [type=missing, input_value={'stand_id': '12345'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
brk_dbh
  Field required [type=missing, input_value={'stand_id': '12345'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing


In [7]:
try:  # negative DBH is invalid
    FvsTreeInitRecord(stand_id="123", species=202, dbh=-1.0)
except ValidationError as e:
    print(e)

1 validation error for FvsTreeInitRecord
dbh
  Input should be greater than 0 [type=greater_than, input_value=-1.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than


You can go back and forth between `FvsStandInit` and `FvsTreeInit` objects and `DataFrame`s using simple helper methods `from_dataframe` and `to_dataframe`. Because MicroFVS is intended to process simulations submitted one stand at a time, each time you create an instance of a `FvsStandInit` or `FvsTreeInit` object from a dataframe, you need to specify the stand ID for the records to retain. This will ignore any data in the input `DataFrame` for other stands.

In [8]:
# Use your favorite data source to create the dataframe
# such as `tree_init_df = pd.read_csv("trees.csv")`

tree_init_df = pd.DataFrame(
    {
        "stand_id": ["12345", "12345", "6789"],
        "plot_id": [1, 1, 1],
        "tree_id": [1, 2, 3],
        "species": ["DF", "WH", "RC"],
        "diameter": [10.5, 1.0, 0.1],
        "tree_count": [150, 25, 25],
    }
)

# notice that the tree from stand_id "6789" is no longer included
FvsTreeInit.from_dataframe(df=tree_init_df, stand_id="12345").to_dataframe()

,stand_id,plot_id,tree_id,tree_count,species,diameter,history,ht,crratio,dg,...,damage3,severity3,tree_value,prescription,slope,aspect,habitat,topo_code,site_prep,age
0,12345,1,1,150.0,DF,10.5,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,12345,1,2,25.0,WH,1.0,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


## 2. FVS Keyfiles and Templating
Under the hood, MicroFVS is executing the Forest Vegetation Simulator as a command-line tool, feeding it a single **FVS Keyfile**. A keyfile is a plain text file that contains a set of **FVS Keywords** representing instructions for FVS to simulate. They are well documented in the FVS [Keyword Reference Guide](https://www.fs.usda.gov/sites/default/files/fvs-keyword.pdf), in the [Fire and Fuels Extension](https://www.fs.usda.gov/sites/default/files/forest-management/fvs-ffe-guide.pdf) and in the [Users Guide to the Database Extension](https://www.fs.usda.gov/sites/default/files/forest-management/fvs-dbs-user-guide.pdf). A set of FVS Keywords is commonly referred to as Keyword Components, and are often saved as `*.kcp` files.

For `MicroFVS`, an `FvsKeyfile` is an object built from an `FvsKeyfileTemplate`, and a set of `FvsKeyfileTemplateParams` to be injected into that template.

In [9]:
FvsKeyfile?

Init signature:
FvsKeyfile(
    *,
    variant: microfvs.enums.FvsVariant,
    stand_id: str,
    treatments: list[microfvs.models.FvsEvent] = [FvsEvent(name='NONE', content='*** NO TREATMENT ***')],
    disturbances: list[microfvs.models.FvsEvent] = [FvsEvent(name='NONE', content='*** NO DISTURBANCE ***')],
    template: str = <FvsKeyfileTemplate.DEFAULT: '********************************************************************************\n** STAND IDENTIFICATION\n********************************************************************************\nSTDIDENT\n{{stand_id}}\n\nDATABASE\nDSNIN\nFVS_Data.db\n\nDSNOUT\nFVS_Data.db\n\nSTANDSQL\nSELECT *\nFROM fvs_standinit\nWHERE stand_id = \'%StandID%\'\nENDSQL\n\nTREESQL\nSELECT *\nFROM fvs_treeinit\nWHERE stand_id = \'%StandID%\'\nENDSQL\n\nEND\n\n********************************************************************************\n** DATABASE Outputs\n** Define database settings for FVS output.\n*****************************************************

We provide a default FVS Keyfile Template. This is a long multi-line string that includes placeholders indicated with double curly braces `"{{placeholder_name}}"` which are used to inject values when the template is rendered. When the template is rendered, each parameter is injected into the placeholder that bears its name. This keyfile template is implemented using the [Jinja2 templating system](https://jinja.palletsprojects.com/en/stable/templates/). 

In [10]:
print(FvsKeyfileTemplate.DEFAULT)

********************************************************************************
** STAND IDENTIFICATION
********************************************************************************
STDIDENT
{{stand_id}}

DATABASE
DSNIN
FVS_Data.db

DSNOUT
FVS_Data.db

STANDSQL
SELECT *
FROM fvs_standinit
WHERE stand_id = '%StandID%'
ENDSQL

TREESQL
SELECT *
FROM fvs_treeinit
WHERE stand_id = '%StandID%'
ENDSQL

END

********************************************************************************
** DATABASE Outputs
** Define database settings for FVS output.
********************************************************************************
DATABASE

** OUTPUT DB TABLES
ATRTLIDB           2         2
BURNREDB           2
CALBSTDB           2
CARBREDB           2
COMPUTDB
CUTLIDB            2         2
DWDCVDB            2
DWDVLDB            2
ECONRPTS           2         2
FUELREDB           2
FUELSOUT           2
INVSTATS
MORTREDB           2         2
POTFIRDB           2
REGREPTS
SNAGOUDB         

The default template provides several placeholders where you can inject all kinds of instructions or settings for the FVS Simulation. Don't worry if you don't have something to fill all the placeholders, there are sensible defaults (e.g., `cycle_length == 5`, `num_cycles == 1`). 

In [11]:
template_params = FvsKeyfileTemplate.DEFAULT.get_template_params()
print(template_params)

Template variables:
  Required: ['stand_id']
  Optional: ['cycle_length', 'disturbance', 'econ', 'first_cycle_length', 'growth_modifiers', 'mortality_modifiers', 'num_cycles', 'sdimax', 'treatment', 'volume']


We feed parameters we want to inject into the template using an `FvsKeyfile` object. The only parameters you must always provide are the `variant` and the `stand_id`. If you don't specify any treatments or disturbances, a "Grow Only" scenario will be generated.

In [12]:
my_simple_keyfile = FvsKeyfile(
    variant=FvsVariant.PN,
    stand_id=12345,
    cycle_length=5,
    num_cycles=10,
)
print(my_simple_keyfile.content)

********************************************************************************
** STAND IDENTIFICATION
********************************************************************************
STDIDENT
12345

DATABASE
DSNIN
FVS_Data.db

DSNOUT
FVS_Data.db

STANDSQL
SELECT *
FROM fvs_standinit
WHERE stand_id = '%StandID%'
ENDSQL

TREESQL
SELECT *
FROM fvs_treeinit
WHERE stand_id = '%StandID%'
ENDSQL

END

********************************************************************************
** DATABASE Outputs
** Define database settings for FVS output.
********************************************************************************
DATABASE

** OUTPUT DB TABLES
ATRTLIDB           2         2
BURNREDB           2
CALBSTDB           2
CARBREDB           2
COMPUTDB
CUTLIDB            2         2
DWDCVDB            2
DWDVLDB            2
ECONRPTS           2         2
FUELREDB           2
FUELSOUT           2
INVSTATS
MORTREDB           2         2
POTFIRDB           2
REGREPTS
SNAGOUDB           2    

Apart from basic configuration settings like the cycle length and number of simulation cycles you want to run, the next most commonly used params will probably be the specification of treatments and disturbance events. We provide a library of treatments and disturbances that you can invoke by name. We have scraped all the examples from a [national compendium of silvicultural treatments](https://doi.org/10.2737/RDS-2022-0037) prepared by a team of silviculture experts from the National Forest System and Research Station of the US Forest Service in the form of FVS Keyword Component (KCP) files, and you can access them by short-hand codes and inject them directly into a template.

In [13]:
library = FvsEventLibrary()
library.usfs_treatments

{'06-06': FvsEvent(name='06-06', content='! Region 6\n! Reference number 6\n!\n! Biomass removal in the Ochoco and Deschutes National Forests\n!\n! Created by Erin Hooten, USDA Forest Service, 2021\n! Updated by Rachel Houtman, Oregon State University, 2022 \n! \n! For complete assignment recommendations and caveats, see: \\Supplements\\SilvicultureCompendium.pdf\n!\n!\n!\n!\n\nSpecPref        2020     Parms(WL, -15.)\nSpecPref        2020     Parms(PP, -10.)\nSpecPref        2020     Parms(DF, -5.)\nSpecPref        2020     Parms(GF, 5.)\nYardLoss        2020        .1         1       .07\n\nthinBTA         2020       40.        1.        0.        9.        0.      999.\n\nFMIN\nPileBurn        2021    Parms(1,100,5,0.7*100,0)\nEND\nFixMort         2021    Parms(All,0.05,0,3.,1,0)'),
 '04-25': FvsEvent(name='04-25', content='! Region 4\n! Reference number 25\n!\n! Even-aged regeneration in Region 4 national forests\n!\n! Created by Erin Hooten, USDA Forest Service, 2021\n! Updated by

To include in new set of keyfile parameters, we'll utilize treatment `06-03`, a commercial thinning prescription developed in USFS Region 6.

In [14]:
COMMERCIAL_THINNING_KEY = "06-03"
thinning_treatment = library.lookup(
    event_type=FvsEventType.TREATMENT,
    event_key=COMMERCIAL_THINNING_KEY,
)
print(thinning_treatment.content)

! Region 6
! Reference number 3
!
! Commercial thin in the Ochoco and Deschutes National Forests
!
! Created by Erin Hooten, USDA Forest Service, 2021
! Updated by Rachel Houtman, Oregon State University, 2022 
! 
! For complete assignment recommendations and caveats, see: \Supplements\SilvicultureCompendium.pdf
!
!
!
!

SpecPref        2020    Parms(PP, -10.)
SpecPref        2020    Parms(DF, -5)
SpecPref        2020    Parms(WJ, 10.)

YardLoss        2020        .1         1       .07

Compute         0
BAHI = SPMCDBH(2, All, 0, 21, 999, 0, 999)
targ = max(0, 40-BAHI)
End

thinBBA         2020       40.        1.        7.       21.        0.      999.

thinBTA         2020        40        1.        0.       6.9        0.      999.

IF                 0
Cut eq Yes
Then
FMIN
PileBurn           1    Parms(1,100,5,0.7*100,0)
END
FixMort            1    Parms(All,0.05,0,3.,1,0)
ENDIF


We'll utilize a disturbance available in the library as well.

In [15]:
INTENSE_FIRE_KEY = "FIC6"
intense_fire = library.lookup(
    event_type=FvsEventType.DISTURBANCE, event_key=INTENSE_FIRE_KEY
)
print(intense_fire.content)

********************************************************************************
** NATURAL DISTURBANCE - WILDFIRE VI (FIC6)
** High-intensity fire implemented in 2035. Fire simulated using a wind speed of
** 38mph, very dry moisture level, temperature of 70F, with FVS-FFE calculating
** mortality. 100% of stand area shall be burned, timed after greenup. Flame
** length modified to have flame length multiplier of 2.0, flame length of 20ft,
** 90% of crowns burning, and a maximum scorch height of 30ft.
********************************************************************************
FMIN
SIMFIRE         2035        38         1        70         1       100         3
FLAMEADJ        2035         2        20        90        30
END



We can now inject this extended set of params into the `FvsKeyfile`. We make a quick helper function here to zoom to the specific segments of the rendered keyfile so you won't have to scroll through the whole thing.

In [ ]:
def zoom_to_pattern(
    text: str, pattern: str, lines_after: int = 15, lines_before: int = 0
) -> str:
    """Zooms to a line matching a pattern and shows what follows it.

    Args:
        text (str): The text body to search.
        pattern (str): The pattern to search for in ``text``.
        lines_after (int): Number of lines to show after the match.
        lines_before (int): Number of lines to show before the match.
    """
    lines = text.splitlines()

    for i, line in enumerate(lines):
        if re.search(pattern, line):
            safe_start = max(0, i - lines_before)
            safe_end = min(len(lines), i + lines_after + 1)
            result = "\n".join(lines[safe_start:safe_end])
            if safe_start > 0:
                result = "...\n" + result
            if safe_end < len(lines):
                result = result + "\n..."
            return result

    return "Pattern not found."

In [17]:
extended_keyfile = FvsKeyfile(
    variant="PN",
    stand_id="12345",
    cycle_length=5,
    num_cycles=10,
    treatments=[thinning_treatment],
    disturbances=[intense_fire],
)

In [18]:
treatment_lines_after = len(thinning_treatment.content.splitlines())
print(
    zoom_to_pattern(
        extended_keyfile.content,
        r"MANAGEMENT TREATMENTS",
        lines_after=treatment_lines_after + 1,
        lines_before=1,
    )
)

...
********************************************************************************
** MANAGEMENT TREATMENTS
********************************************************************************
! Region 6
! Reference number 3
!
! Commercial thin in the Ochoco and Deschutes National Forests
!
! Created by Erin Hooten, USDA Forest Service, 2021
! Updated by Rachel Houtman, Oregon State University, 2022 
! 
! For complete assignment recommendations and caveats, see: \Supplements\SilvicultureCompendium.pdf
!
!
!
!

SpecPref        2020    Parms(PP, -10.)
SpecPref        2020    Parms(DF, -5)
SpecPref        2020    Parms(WJ, 10.)

YardLoss        2020        .1         1       .07

Compute         0
BAHI = SPMCDBH(2, All, 0, 21, 999, 0, 999)
targ = max(0, 40-BAHI)
End

thinBBA         2020       40.        1.        7.       21.        0.      999.

thinBTA         2020        40        1.        0.       6.9        0.      999.

IF                 0
Cut eq Yes
Then
FMIN
PileBurn           1 

In [19]:
disturbance_lines_after = len(intense_fire.content.splitlines())
print(
    zoom_to_pattern(
        extended_keyfile.content,
        r"NATURAL DISTURBANCES",
        lines_after=disturbance_lines_after + 2,
        lines_before=1,
    )
)

...
********************************************************************************
** NATURAL DISTURBANCES
********************************************************************************
********************************************************************************
** NATURAL DISTURBANCE - WILDFIRE VI (FIC6)
** High-intensity fire implemented in 2035. Fire simulated using a wind speed of
** 38mph, very dry moisture level, temperature of 70F, with FVS-FFE calculating
** mortality. 100% of stand area shall be burned, timed after greenup. Flame
** length modified to have flame length multiplier of 2.0, flame length of 20ft,
** 90% of crowns burning, and a maximum scorch height of 30ft.
********************************************************************************
FMIN
SIMFIRE         2035        38         1        70         1       100         3
FLAMEADJ        2035         2        20        90        30
END

...


### Roll Your Own
You are welcome to create your own keyfile templates. We demonstrate that here for thoroughness, although we'd strongly encourage you to try and work with the default template first, which we've designed to be very flexible. 

Making your own template is relatively simple to get started, but if you're not careful to follow the spacing requirements, you might introduce unexpected bugs into the simulation. And FVS may or may not give you useful error messages.

In the example below, there are no commands to read in stand or tree data, FVS simulation settings configured, and no call to FVS to process the keyfile. This is not a runnable keyfile, but it can demonstrate the templating process, including the use of nested placeholders within keyword components.

In [20]:
CUSTOM_KEYFILE_TEMPLATE = """
** {{ helpful_comment }}
{{ some_keywords }}
{{ keywords_with_nested_placeholder }}
** {{ unhelpful_comment }}
"""

STATIC_KEYWORD_COMPONENTS = """
** This demonstrates injection of static keyword components.
COMPUTE            0
DF_TPA = SPMCDBH(1,DF,0)
END
"""

KEYWORDS_WITH_NESTED_PLACEHOLDER = """
** This demonstrates the use of a nested placeholder.
COMPUTE            0
{{ nested_placeholder_1 }} = SPMCDBH(1,{{ nested_placeholder_2 }},0)
END
"""

custom_keyfile = FvsKeyfile(
    variant="PN",
    stand_id="87654",
    template=CUSTOM_KEYFILE_TEMPLATE,
    template_params={
        "helpful_comment": "Custom Keyfile version 0.1.0",
        "some_keywords": STATIC_KEYWORD_COMPONENTS,
        "keywords_with_nested_placeholder": KEYWORDS_WITH_NESTED_PLACEHOLDER,
        "unhelpful_comment": "I wonder what this button does.",
        "nested_placeholder_1": "RC_TPA",
        "nested_placeholder_2": "RC",
    },
)

print(custom_keyfile.content)


** Custom Keyfile version 0.1.0

** This demonstrates injection of static keyword components.
COMPUTE            0
DF_TPA = SPMCDBH(1,DF,0)
END


** This demonstrates the use of a nested placeholder.
COMPUTE            0
RC_TPA = SPMCDBH(1,RC,0)
END

** I wonder what this button does.


Instead of rolling your own keyfile template, we'd generally encourage you to generate keyword components that can be injected into the default template. 

For example, if you wanted to enter a custom SDIMAX keyword, you could do it like this.

In [21]:
MY_SDIMAX = "SDIMAX           All       600"

You might want to be very careful making raw strings though, because FVS is particularly about the spacing of keywords and their arguments. It's safer to enforce spacing and column alignment of keyword arguments using an approach like this.

In [22]:
def make_fixed_width_keyword(
    keyword,
    f1=None,
    f2=None,
    f3=None,
    f4=None,
    f5=None,
    f6=None,
    f7=None,
):
    """A simplistic function to make fixed-width keywords."""
    fs = {}
    for i, f in enumerate([f1, f2, f3, f4, f5, f6, f7]):
        fs[i] = f or ""
    return (
        f"{keyword:<10}{fs[0]:>10}{fs[1]:>10}{fs[2]:>10}{fs[3]:>10}"
        f"{fs[4]:>10}{fs[5]:>10}{fs[6]:>10}"
    )


df_sdimax = make_fixed_width_keyword(keyword="SDIMAX", f1="DF", f2=600)
wh_sdimax = make_fixed_width_keyword(keyword="SDIMAX", f1="WH", f2=1000)
my_sdimax_kcp = "\n".join([df_sdimax, wh_sdimax])

Scroll down to see your custom keyword component injected at the `sdimax` placeholder we had in the 

In [23]:
custom_sdimax_keyfile = FvsKeyfile(
    variant="PN",
    stand_id="87654",
    template_params={"sdimax": my_sdimax_kcp},
)

print(
    zoom_to_pattern(
        custom_sdimax_keyfile.content,
        r"SDIMAX SETTINGS",
        lines_after=len(my_sdimax_kcp.splitlines()) + 1,
        lines_before=1,
    )
)

...
********************************************************************************
** SDIMAX SETTINGS
********************************************************************************
SDIMAX            DF       600                                                  
SDIMAX            WH      1000                                                  
...


## 3. Interacting with the MicroFVS REST API
Now we're ready to see how we can interact with MicroFVS as a web service using `GET` and `POST` requests. As we develop client libraries for interacting with MicroFVS through Python and R, we expect to simplify these use cases substantially so that users are not expected to specify `GET` and `POST` requests themselves, nor to parse results back into formats like DataFrames. 

We will demonstrate a few key endpoints of the MicroFVS REST API to which requests can be submitted:
* **Version Check:** Which versions are being run for each FVS variant in the web service?
* **Run FVS Simulation:** Execute FVS with your own inventory data, using the default keyfile template and grow-only scenario.
* **Run FVS with Treatment & Disturbance:** Insert a thinning treatment before an intense wildfire.


In [24]:
MICROFVS_BASE_URL = "http://localhost:8000"
TEMPLATE_ENDPOINT = MICROFVS_BASE_URL + "/template"
VERSION_ENDPOINT = MICROFVS_BASE_URL + "/version"
KEYFILE_ENDPOINT = MICROFVS_BASE_URL + "/keyfile"
RUN_ENDPOINT = MICROFVS_BASE_URL + "/run"

### Check the FVS Versions
In this first instance, we use a `GET` request to the `/versions` endpoint. The result is a dictionary

In [25]:
response = requests.get(VERSION_ENDPOINT)
content = response.content.decode()
json_content = json.loads(content)
pretty_json = json.dumps(json_content, indent=4)
print(pretty_json)

{
    "Southeast Alaska and Coastal British Columbia (AK)": "RV:20250930",
    "Blue Mountains (BM)": "RV:20250930",
    "Inland California and Southern Cascades (ICASCA) (CA)": "RV:20250930",
    "Central Idaho (CI)": "RV:20250930",
    "Central Rockies (CR)": "RV:20250930",
    "Central States (CS)": "RV:20250930",
    "East Cascades (EC)": "RV:20250930",
    "Eastern Montana (EM)": "RV:20250930",
    "Inland Empire (IE)": "RV:20250930",
    "Kootenai, Kaniksu, and Tally Lake (KooKanTL) (KT)": "RV:20250930",
    "Lake States (LS)": "RV:20250930",
    "Klamath Mountains (and northern California) (NC)": "RV:20250930",
    "Northeastern US (NE)": "RV:20250930",
    "Organon Southwest (OC)": "RV:20250930",
    "Organon Pacific Northwest (OP)": "RV:20250930",
    "Pacific Northwest Coast (PN)": "RV:20250930",
    "Southern US (SN)": "RV:20250930",
    "South Central Oregon and Northeast California (SORNEC) (SO)": "RV:20250930",
    "Tetons (TT)": "RV:20250930",
    "Utah (UT)": "RV:202509

### Executing FVS (grow only)
Here we demonstrate a couple simulations triggered using a `POST` request to the `/run` endpoint. 

We construct some simple stand and tree initialization data here, but you could also utilize the `FvsStandInit.from_dataframe` and `FvsTreeInit.from_dataframe` approaches with any datasets you can load into dataframe format. 

In [26]:
stand_init = FvsStandInit(
    stand_id="12345",
    variant="PN",
    inv_year=2020,
    basal_area_factor=-10,  # 1/10th-acre fixed radius plot for large trees
    inv_plot_size=100,  # 1/100th-acre fixed radius plot for small trees
    brk_dbh=5,  # large trees start at 5" DBH
)

tree1 = FvsTreeInitRecord(
    stand_id="12345",
    plot_id=1,
    tree_id=1,
    species="DF",
    diameter=14.5,
    tree_count=6,
)
tree2 = FvsTreeInitRecord(
    stand_id="12345",
    plot_id=1,
    tree_id=2,
    species="WH",
    dbh=12.0,
    tree_count=3,
)
seed1 = FvsTreeInitRecord(
    stand_id="12345", plot_id=1, tree_id=3, species="RC", dbh=0.1, tree_count=2
)
tree_init = FvsTreeInit(trees=[tree1, tree2, seed1])

The response from the `/run` endpoint is an instance of `FvsResult` in JSON format. `FvsResult` combines metadata about the simulation, as well as the output data. 

We have scraped some fundamental metadata from these simulations beyond what FVS itself usually provides in output forms to help clearly document the computation and to make it more reproducible and auditable: 
* **`command`:** the command-line call that was issued to trigger the FVS simulation.
* **`return_code`:** the return code from the command line which indicates the severity of problems (if any) that FVS encountered during hte simulation.
* **`stdout`:** the Standard Output echoed to the terminal during the FVS simulation.
* **`stderr`:** the Standard Error echoed to the terminal during the FVS simulation, which may indicate when FVS experiences a catastrophic failure (e.g., a segmentation fault), in which case the resulting outputs may be fully or partially incomplete.
* **`fvs_warnings`:** FVS warnings and errors that were detected in the plain text output file.

In [27]:
basic_run_response = requests.post(
    RUN_ENDPOINT,
    json={
        "stand_init": stand_init.model_dump(),
        "tree_init": tree_init.model_dump(),
        "template_params": {
            "num_cycles": 10,
            "cycle_length": 5,
            # ... other parameters you want to inject into the template
        },
    },
)
basic_run_result = FvsResult.model_validate(basic_run_response.json())
print(basic_run_result)

FvsResult(
	name: PN_12345_NONE_NONE,
	fvs_variant: PN,
	stand_id: 12345,
	treatment: NONE,
	disturbance: NONE,
	command: /usr/local/bin/FVSpn --keywordfile=PN_12345_NONE_NONE.key,
	return_code: 10,
	stdout: None,
	stderr: None,
	fvs_warnings: [
		line_number=45 type='WARNING' id='FVS03' message='FOREST CODE INDICATES THE GEOGRAPHIC LOCATION IS OUTSIDE THE RANGE OF THE MODEL. DEFAULT CODE IS USED.',
		line_number=323 type='WARNING' id='FVS14' message='HABITAT/PLANT ASSOCIATION/ECOREGION CODE WAS NOT RECOGNIZED; HABITAT/PLANT ASSOCIATION/ECOREGION SET TO DEFAULT CODE.',
	]
	fvs_data: {
		fvs_standinit: (1 rows, 63 columns),
		fvs_treeinit: (3 rows, 26 columns),
		fvs_cases: (1 rows, 12 columns),
		fvs_error: (2 rows, 3 columns),
		fvs_calibstats: (0 rows, 0 columns),
		fvs_stats_species: (3 rows, 12 columns),
		fvs_invreference: (38 rows, 20 columns),
		fvs_treelist: (255 rows, 35 columns),
		fvs_strclass: (21 rows, 46 columns),
		fvs_snagdet: (146 rows, 17 columns),
		fvs_snagsum: (10 

This `print(basic_run_result)` above just shows a print-friendly summary of the `FvsResult`. The two other attributes you might be interested in inspecting include the content of the plain-text keyfile that was used to execute the simulation...

In [28]:
print(basic_run_result.keyfile)

********************************************************************************
** STAND IDENTIFICATION
********************************************************************************
STDIDENT
12345

DATABASE
DSNIN
FVS_Data.db

DSNOUT
FVS_Data.db

STANDSQL
SELECT *
FROM fvs_standinit
WHERE stand_id = '%StandID%'
ENDSQL

TREESQL
SELECT *
FROM fvs_treeinit
WHERE stand_id = '%StandID%'
ENDSQL

END

********************************************************************************
** DATABASE Outputs
** Define database settings for FVS output.
********************************************************************************
DATABASE

** OUTPUT DB TABLES
ATRTLIDB           2         2
BURNREDB           2
CALBSTDB           2
CARBREDB           2
COMPUTDB
CUTLIDB            2         2
DWDCVDB            2
DWDVLDB            2
ECONRPTS           2         2
FUELREDB           2
FUELSOUT           2
INVSTATS
MORTREDB           2         2
POTFIRDB           2
REGREPTS
SNAGOUDB           2    

... as well as the plain-text outfile that FVS generated. This output may be particularly useful for debugging problematic simulations, since it will often indicate where in the sequence of simulation instructions an error or warning was encountered.

In [29]:
print(basic_run_result.outfile)



     FOREST VEGETATION SIMULATOR     VERSION FS2025.4 -- PACIFIC NW COAST PROGNOSIS             RV:20250930    04-09-2026  19:48:05

----------------------------------------------------------------------------------------------------------------------------------

                                                OPTIONS SELECTED BY INPUT

KEYWORD FILE NAME: PN_12345_NONE_NONE
----------------------------------------------------------------------------------------------------------------------------------
KEYWORD    PARAMETERS:
--------   -----------------------------------------------------------------------------------------------------------------------


           ********************************************************************************
           ** STAND IDENTIFICATION
           ********************************************************************************

STDIDENT
           STAND ID= 12345                                                                              

While some FVS nerds enjoy nothing more than curling up in a comfy chair beside the fireplace to review plain-text input and output files, most well-adjusted users of the API will no doubt be more interested in the tabular output data generated by FVS. The `FvsResult.fvs_data` is the gateway to all of the data tables we scraped from the FVS database which was created on the web server to hold both the FVS inventory data inputs and the simulation outputs. The data tables are stored in a dictionary, and you can access them by name (e.g., `result.fvs_data["fvs_summary2"]`). 

The MicroFVS API receives and returns data as JSON-formatted strings. JSON does not include a native specification for tabular data like a dataframe. The data tables scraped from the FVS output database were converted into a list of records (or rows). We reconstruct the output data into a dataframe using `pd.DataFrame.from_records(...)`.

As we continue to develop the client libraries for interacting with MicroFVS, we will make it easier so you don't have to handle these types of format-conversions yourself.

In [30]:
basic_run_result.fvs_data[FvsOutputTableName.FVS_SUMMARY2]

,caseid,standid,year,rmvcode,age,tpa,tprdtpa,ba,sdi,ccf,...,rmcuft,rscuft,rbdft,prdlen,acc,mort,mai,fortyp,sizecls,stkcls
0,23c0f4a9-5324-4eb5-adee-7149bd530ead,12345,2020,0,0,290.000000,290.000000,92.377014,188,111,...,0.0,0.0,0.0,5,281.249573,3.143651,0.0,201,1,3
1,23c0f4a9-5324-4eb5-adee-7149bd530ead,12345,2025,0,5,273.277863,273.277863,129.155273,244,151,...,0.0,0.0,0.0,5,296.991272,4.531378,0.0,201,1,3
2,23c0f4a9-5324-4eb5-adee-7149bd530ead,12345,2030,0,10,257.217377,257.217377,166.579010,295,191,...,0.0,0.0,0.0,5,305.840912,6.069248,0.0,201,1,2
3,23c0f4a9-5324-4eb5-adee-7149bd530ead,12345,2035,0,15,241.869934,241.869934,202.958115,342,226,...,0.0,0.0,0.0,5,307.154297,7.701991,0.0,201,1,2
4,23c0f4a9-5324-4eb5-adee-7149bd530ead,12345,2040,0,20,227.229950,227.229950,237.727524,383,259,...,0.0,0.0,0.0,5,303.266693,9.393149,0.0,201,1,2
5,23c0f4a9-5324-4eb5-adee-7149bd530ead,12345,2045,0,25,213.314774,213.314774,269.875336,419,288,...,0.0,0.0,0.0,5,315.178497,11.072767,0.0,201,1,2
6,23c0f4a9-5324-4eb5-adee-7149bd530ead,12345,2050,0,30,200.179703,200.179703,302.394318,453,318,...,0.0,0.0,0.0,5,294.275360,12.913610,0.0,201,1,2
7,23c0f4a9-5324-4eb5-adee-7149bd530ead,12345,2055,0,35,187.801804,187.801804,330.835846,481,344,...,0.0,0.0,0.0,5,283.509766,18.504353,0.0,201,1,1
8,23c0f4a9-5324-4eb5-adee-7149bd530ead,12345,2060,0,40,135.908798,135.908798,354.273163,477,360,...,0.0,0.0,0.0,5,281.593994,18.352436,0.0,201,1,1
9,23c0f4a9-5324-4eb5-adee-7149bd530ead,12345,2065,0,45,108.398849,108.398849,378.370605,481,379,...,0.0,0.0,0.0,5,267.383545,18.748123,0.0,201,1,1


### A More Eventful FVS Simulation
Here we will employ a couple events from the MicroFVS EventLibrary demonstrated above, the commercial thinning treatment defined by the USFS and the the intense wildfire event defined by Vibrant Planet.

In [31]:
# build a new set of params with the added treatment and disturbance
eventful_template_params = {
    "num_cycles": 10,
    "cycle_length": 5,
    "treatments": [COMMERCIAL_THINNING_KEY],
    "disturbances": [INTENSE_FIRE_KEY],
}

# submit the simulation parameters to the MicroFVS web service
eventful_run_response = requests.post(
    RUN_ENDPOINT,
    json={
        "stand_init": stand_init.model_dump(),
        "tree_init": tree_init.model_dump(),
        "template_params": eventful_template_params,
    },
)
# parse the response from JSON back into an FvsResult object
eventful_run_result = FvsResult.model_validate(eventful_run_response.json())
print(eventful_run_result)

FvsResult(
	name: PN_12345_06-03_FIC6,
	fvs_variant: PN,
	stand_id: 12345,
	treatment: 06-03,
	disturbance: FIC6,
	command: /usr/local/bin/FVSpn --keywordfile=PN_12345_06-03_FIC6.key,
	return_code: 10,
	stdout: None,
	stderr: None,
	fvs_warnings: [
		line_number=45 type='WARNING' id='FVS03' message='FOREST CODE INDICATES THE GEOGRAPHIC LOCATION IS OUTSIDE THE RANGE OF THE MODEL. DEFAULT CODE IS USED.',
		line_number=406 type='WARNING' id='FVS14' message='HABITAT/PLANT ASSOCIATION/ECOREGION CODE WAS NOT RECOGNIZED; HABITAT/PLANT ASSOCIATION/ECOREGION SET TO DEFAULT CODE.',
	]
	fvs_data: {
		fvs_standinit: (1 rows, 63 columns),
		fvs_treeinit: (3 rows, 26 columns),
		fvs_cases: (1 rows, 12 columns),
		fvs_error: (2 rows, 3 columns),
		fvs_calibstats: (0 rows, 0 columns),
		fvs_stats_species: (3 rows, 12 columns),
		fvs_invreference: (38 rows, 20 columns),
		fvs_treelist: (126 rows, 35 columns),
		fvs_strclass: (21 rows, 46 columns),
		fvs_cutlist: (3 rows, 35 columns),
		fvs_atrtlist: (2

You can inspect the result's keyfile attribute if you want to confirm that the additional instructions we added got inserted. You can also just look in the FVS output tables and see that harvest removals did occur in 2020 and a fire was triggered in 2035.

In [32]:
# print(eventful_run_result.keyfile)
eventful_run_result.fvs_data[FvsOutputTableName.FVS_SUMMARY2]

,caseid,standid,year,rmvcode,age,tpa,tprdtpa,ba,sdi,ccf,...,rmcuft,rscuft,rbdft,prdlen,acc,mort,mai,fortyp,sizecls,stkcls
0,7e9bfca1-44c1-44dd-b3ab-cf287045002b,12345,2020,1,0,290.000000,290.000000,92.377014,188,111,...,1412.924072,0.0,7229.456055,5,136.400757,1.402896,0.0,201,1,4
1,7e9bfca1-44c1-44dd-b3ab-cf287045002b,12345,2020,2,0,74.881622,290.000000,40.002190,74,49,...,0.000000,0.0,0.000000,5,136.400757,1.402896,0.0,201,1,4
2,7e9bfca1-44c1-44dd-b3ab-cf287045002b,12345,2025,0,5,70.188965,285.307343,58.630413,99,69,...,0.000000,0.0,0.000000,5,147.230499,2.076525,0.0,201,1,4
3,7e9bfca1-44c1-44dd-b3ab-cf287045002b,12345,2030,0,10,67.674095,282.792473,78.354874,124,89,...,0.000000,0.0,0.000000,5,160.210587,2.831523,0.0,201,1,4
4,7e9bfca1-44c1-44dd-b3ab-cf287045002b,12345,2035,0,15,65.319519,280.437897,99.158394,149,110,...,0.000000,0.0,0.000000,5,15.740543,651.001404,0.0,201,5,5
5,7e9bfca1-44c1-44dd-b3ab-cf287045002b,12345,2040,0,20,3.246327,218.364705,11.244080,14,12,...,0.000000,0.0,0.000000,5,16.567528,0.416637,0.0,999,5,5
6,7e9bfca1-44c1-44dd-b3ab-cf287045002b,12345,2045,0,25,3.226549,218.344927,13.236019,16,14,...,0.000000,0.0,0.000000,5,16.964869,0.496831,0.0,999,5,5
7,7e9bfca1-44c1-44dd-b3ab-cf287045002b,12345,2050,0,30,3.210449,218.328827,15.223940,18,16,...,0.000000,0.0,0.000000,5,17.060654,0.579005,0.0,999,5,5
8,7e9bfca1-44c1-44dd-b3ab-cf287045002b,12345,2055,0,35,3.194429,218.312807,17.197662,20,18,...,0.000000,0.0,0.000000,5,17.179100,0.661247,0.0,999,5,5
9,7e9bfca1-44c1-44dd-b3ab-cf287045002b,12345,2060,0,40,3.178489,218.296867,19.150023,22,19,...,0.000000,0.0,0.000000,5,17.225332,0.743670,0.0,999,5,5


In [33]:
eventful_run_result.fvs_data[FvsOutputTableName.FVS_CONSUMPTION]

,caseid,standid,year,min_soil_exp,litter_consumption,duff_consumption,consumption_lt3,consumption_ge3,consumption_3to6,consumption_6to12,consumption_ge12,consumption_herb_shrub,consumption_crowns,total_consumption,percent_consumption_duff,percent_consumption_ge3,percent_trees_crowning,smoke_production_25,smoke_production_10
0,7e9bfca1-44c1-44dd-b3ab-cf287045002b,12345,2035,60.521683,0.861665,14.404565,1.268812,5.506289,2.246796,2.111995,1.147498,0.765891,3.036044,25.843267,77.309998,62.408829,90,0.280332,0.330338


In [34]:
eventful_run_result.fvs_data[FvsOutputTableName.FVS_BURN_REPORT]

,caseid,standid,year,one_hr_moisture,ten_hr_moisture,hundred_hr_moisture,thousand_hr_moisture,duff_moisture,live_woody_moisture,live_herb_moisture,...,scorch_height,fire_type,fuelmodl1,weight1,fuelmodl2,weight2,fuelmodl3,weight3,fuelmodl4,weight4
0,7e9bfca1-44c1-44dd-b3ab-cf287045002b,12345,2035,4.0,4.0,5.0,10.0,15.000001,70.0,70.0,...,30.0,USER_DEF,5,95.0,10,5.0,0,0.0,0,0.0


### Passing a custom set of treatments and/or disturbance events.

If we want to pass on or more custom treatments or disturbances, we need to provide them with a name and with the keyword components as content.

In [35]:
thin_sdi_2030_kcp = """
*** In 2030, thin throughout the diameter range to target SDI of 200. ***
THINSDI         2030       200
"""
thin_sdi_2050_kcp = """
*** In 2050, thin throughout the diameter range to target SDI of 250. ***
THINSDI         2050       250
"""

custom_keyfile_params = {
    "num_cycles": 10,
    "cycle_length": 5,
    # for custom events,
    # we need to specify the names and contents for each event
    "treatments": [
        {"name": "thin-sdi-2030", "content": thin_sdi_2030_kcp},
        {"name": "thin-sdi-2050", "content": thin_sdi_2050_kcp},
    ],
}

custom_run_response = requests.post(
    RUN_ENDPOINT,
    json={
        "stand_init": stand_init.model_dump(),
        "tree_init": tree_init.model_dump(),
        "template_params": custom_keyfile_params,
    },
)
custom_run_result = FvsResult.model_validate(custom_run_response.json())
print(custom_run_result)

FvsResult(
	name: PN_12345_thin-sdi-2030+thin-sdi-2050_NONE,
	fvs_variant: PN,
	stand_id: 12345,
	treatment: thin-sdi-2030+thin-sdi-2050,
	disturbance: NONE,
	command: /usr/local/bin/FVSpn --keywordfile=PN_12345_thin-sdi-2030+thin-sdi-2050_NONE.key,
	return_code: 10,
	stdout: None,
	stderr: None,
	fvs_warnings: [
		line_number=45 type='WARNING' id='FVS03' message='FOREST CODE INDICATES THE GEOGRAPHIC LOCATION IS OUTSIDE THE RANGE OF THE MODEL. DEFAULT CODE IS USED.',
		line_number=342 type='WARNING' id='FVS14' message='HABITAT/PLANT ASSOCIATION/ECOREGION CODE WAS NOT RECOGNIZED; HABITAT/PLANT ASSOCIATION/ECOREGION SET TO DEFAULT CODE.',
	]
	fvs_data: {
		fvs_standinit: (1 rows, 63 columns),
		fvs_treeinit: (3 rows, 26 columns),
		fvs_cases: (1 rows, 12 columns),
		fvs_error: (2 rows, 3 columns),
		fvs_calibstats: (0 rows, 0 columns),
		fvs_stats_species: (3 rows, 12 columns),
		fvs_invreference: (38 rows, 20 columns),
		fvs_treelist: (255 rows, 35 columns),
		fvs_strclass: (21 rows, 46

We can see that the thins were triggered when we called for them, and that the after-thinning SDI matches the targets we specified (check out the rows with `rmvcode == 2` to see after-treatment values and `rmvcode == 1` for the before-treatment values).

In [36]:
DISPLAY_COLS = [
    "standid",
    "year",
    "rmvcode",
    "tpa",
    "ba",
    "qmd",
    "sdi",
    "bdft",
    "rbdft",
]
custom_run_result.fvs_data[FvsOutputTableName.FVS_SUMMARY2][DISPLAY_COLS]

,standid,year,rmvcode,tpa,ba,qmd,sdi,bdft,rbdft
0,12345,2020,0,290.000000,92.377014,7.642215,188,14730.000000,0.000000
1,12345,2025,0,273.277863,129.155273,9.308723,244,21528.039062,0.000000
2,12345,2030,1,257.217377,166.579010,10.896732,295,30179.765625,9734.862305
3,12345,2030,2,174.248672,112.846848,10.896731,200,20444.903320,0.000000
4,12345,2035,0,165.310028,139.966461,12.459444,235,27260.636719,0.000000
5,12345,2040,0,156.881897,166.392136,13.944927,268,33003.050781,0.000000
6,12345,2045,0,148.933136,191.891068,15.369793,297,40880.781250,0.000000
7,12345,2050,1,126.309326,217.009155,17.748333,317,48962.226562,10372.839844
8,12345,2050,2,99.550201,171.034912,17.748335,250,38589.386719,0.000000
9,12345,2055,0,89.456352,190.586136,19.764069,267,44287.675781,0.000000
